# Lab 8, optional part: PyTorch

This notebook introduces PyTorch through the algorithms of Lab 8. The first three sections
cover tensors, automatic differentiation and the training loop; the fourth trains a neural
policy with REINFORCE on the cart-pole, and the fifth a deep Q-network (DQN) on the falling
blocks.

Replace the lines `...  # TODO` with your code.

PyTorch is not part of the course environment. Install the CPU version once, in a terminal
where the environment is active:
`pip install torch --index-url https://download.pytorch.org/whl/cpu` (about 200 MB), or
use the Colab version of this notebook, where PyTorch is already installed.

In [ ]:
# Checks that your folder seqlearn/ is recent enough, and updates it otherwise. (Provided, just run it.)
import importlib.util, io, pathlib, re, shutil, sys, urllib.request, zipfile

REQUIRED = "2026.11.03"
URL = "https://igorcolin.github.io/teaching/sequential-decision/sequential-decision-setup.zip"


def seqlearn_version():
    spec = importlib.util.find_spec("seqlearn")
    if spec is None or spec.origin is None:
        return None, pathlib.Path("seqlearn")
    init = pathlib.Path(spec.origin)
    match = re.search(r'^__version__ = "([^"]+)"', init.read_text(), re.MULTILINE)
    return (match.group(1) if match else "none"), init.parent


version, folder = seqlearn_version()
if version is not None and version != "none" and version >= REQUIRED:
    print(f"seqlearn {version}: up to date")
else:
    print(f"seqlearn {version or 'not found'}: version {REQUIRED} or later needed, downloading it")
    try:
        data = urllib.request.urlopen(URL, timeout=30).read()
    except OSError as error:
        raise RuntimeError(f"download failed ({error}): download {URL} by hand, unzip it, "
                           "and replace your folder seqlearn/ with the new one") from None
    old = folder.with_name("seqlearn.old")
    if folder.exists():
        shutil.rmtree(old, ignore_errors=True)
        folder.rename(old)
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        for name in archive.namelist():
            parts = pathlib.PurePosixPath(name).parts
            if len(parts) > 2 and parts[1] == "seqlearn" and not name.endswith("/"):
                target = folder.joinpath(*parts[2:])
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(name))
    importlib.invalidate_caches()
    version = seqlearn_version()[0]
    if version == "none" or version < REQUIRED:
        raise RuntimeError(f"the course page still has seqlearn {version}: tell your teacher")
    print(f"seqlearn updated to {version} (previous folder kept as {old})")
    if "seqlearn" in sys.modules:
        print("Restart the kernel (Kernel > Restart), then run the notebook from the top.")

In [ ]:
import copy

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

from seqlearn.blocks import Blocks
from seqlearn.control import CartPole
from seqlearn.plotting import plot_mean, set_style

set_style()
torch.set_num_threads(1)  # small networks: one thread is faster

## 1. Tensors

A tensor is a numpy array that can live on a GPU and remember how it was computed. Most
numpy functions have the same name: `torch.zeros`, `torch.exp`, `x.sum(dim=1)` (instead of
`axis=1`), `x @ y`. `torch.from_numpy(a)` and `x.numpy()` convert, without copying. The
default type is `float32`, and two tensors in an operation must have the same type.

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0], [0.0, -1.0, 1.0]])
print(x.shape, x.dtype)
print(x.sum(dim=1), x.max(dim=1).values, x.T @ x)

**Question 1.** Write `softmax_rows(z)`, the softmax of each row of a 2-d tensor, with
`torch.exp` and `sum` only (subtract the maximum of each row first), and compare with
`torch.softmax(z, dim=1)`.

In [ ]:
def softmax_rows(z):
    ...  # TODO


z = torch.randn(4, 3) * 10
print(softmax_rows(z))
print("same as torch.softmax:", torch.allclose(softmax_rows(z), torch.softmax(z, dim=1)))

## 2. Automatic differentiation

A tensor created with `requires_grad=True` records the operations applied to it. Calling
`.backward()` on a scalar computed from it fills its attribute `.grad` with the gradient.
This is how the gradients of networks are computed: nobody differentiates by hand.

**Question 2.** For a random $2 \times 4$ matrix $W$, a random state $s$ and $a = 1$,
compute $\log \pi_W(a \mid s)$ with `torch.log_softmax`, call `backward`, and compare
`W.grad` with the formula $(e_a - \pi_W(\cdot \mid s))\, s^\top$ of Lab 8.

In [ ]:
torch.manual_seed(0)
W = torch.randn(2, 4, requires_grad=True)
s = torch.randn(4)
a = 1
...  # TODO

Gradients add up: a second `backward` adds to `W.grad`, which must be reset to zero
between two steps. `with torch.no_grad():` turns the recording off, for computations that
are not differentiated (updates, evaluation).

## 3. Modules and optimizers

A network is a module: `nn.Linear(d, k)` computes $x \mapsto Ax + b$, `nn.Sequential`
chains modules, and `net.parameters()` lists the tensors to learn. An optimizer updates
them from their `.grad`. Every training loop has the same four lines:

```python
loss = ...              # a scalar computed from the parameters
optimizer.zero_grad()   # reset the gradients
loss.backward()         # compute them
optimizer.step()        # update the parameters
```

**Question 3.** Complete the training loop of a logistic regression on the two clouds of
points below, with the loss `nn.BCEWithLogitsLoss()` and `torch.optim.SGD` (step $0.1$),
for $200$ steps on the whole data set. Print the accuracy at the end.

In [ ]:
rng = np.random.default_rng(0)
X = torch.from_numpy(np.vstack([rng.normal([-1, 0], 1, (200, 2)), rng.normal([1, 1], 1, (200, 2))]).astype(np.float32))
y = torch.cat([torch.zeros(200), torch.ones(200)])

model = nn.Linear(2, 1)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
for step in range(200):
    ...  # TODO
with torch.no_grad():
    accuracy = ((model(X).squeeze(1) > 0).float() == y).float().mean()
print(f"loss {loss.item():.3f}, accuracy {accuracy.item():.3f}")

## 4. REINFORCE with a neural policy

The policy is now a network with one hidden layer, which outputs the two logits. As in the
lecture, REINFORCE minimizes the surrogate loss
$$L = -\sum_t (G_t - b)\, \log \pi(A_t \mid S_t) \enspace,$$
whose gradient is minus the REINFORCE estimate. Here $b$ is the mean of the $G_t$ of the
episode.

**Question 4.** Complete `reinforce_loss(policy, states, actions, G)`, where `states` is a
tensor of shape $(T, 4)$, `actions` a tensor of integers of shape $(T,)$ and `G` the
returns-to-go. Then train $3$ policies for $300$ episodes with Adam (step $0.01$), one
update per episode, and plot the mean length of the episodes.

In [ ]:
def reinforce_loss(policy, states, actions, G):
    ...  # TODO


def returns_to_go(rewards, gamma):
    G, acc = np.zeros(len(rewards), dtype=np.float32), 0.0
    for t in range(len(rewards) - 1, -1, -1):
        acc = rewards[t] + gamma * acc
        G[t] = acc
    return G


def train_reinforce(seed, n_episodes=300, gamma=0.99, lr=0.01):
    torch.manual_seed(seed)
    rng, env = np.random.default_rng(seed), CartPole(rng=seed)
    policy = nn.Sequential(nn.Linear(4, 32), nn.Tanh(), nn.Linear(32, 2))
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    lengths = []
    for _ in range(n_episodes):
        states, actions, rewards = [], [], []
        s, done = env.reset(), False
        while not done:
            with torch.no_grad():
                p = torch.softmax(policy(torch.as_tensor(s, dtype=torch.float32)), dim=0).numpy()
            a = int(rng.choice(2, p=p / p.sum()))
            states.append(s)
            actions.append(a)
            s, r, done = env.step(a)
            rewards.append(r)
        loss = reinforce_loss(policy, torch.as_tensor(np.array(states), dtype=torch.float32),
                              torch.as_tensor(actions), torch.from_numpy(returns_to_go(rewards, gamma)))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lengths.append(len(rewards))
    return np.array(lengths)


...  # TODO

## 5. A deep Q-network on the falling blocks

In Lab 8, $Q_w(s, a)$ was a linear function of features chosen by hand. Here
$Q(s, a)$ is a network that reads the raw board obtained after the placement ($60$ cells,
$0$ or $1$) and the number of rows removed: it must find the useful features itself. As in
the lecture, two devices make the semi-gradient updates stable:

- a **replay buffer** keeps the last $20000$ transitions, and each update uses a
  mini-batch of $32$ transitions drawn at random among them;
- a **target network**, a copy of the network updated every $500$ steps, computes the
  targets $y = r + \gamma \max_{a'} Q^-(s', a')$.

`inputs(game, board, piece)` returns the placements, the inputs of the network for each of
them, their rewards and the boolean array of the placements that end the game, as
`placement_features` in Lab 8.

In [ ]:
def inputs(game, board, piece):
    actions = game.actions(piece)
    X = np.zeros((len(actions), game.height * game.width + 1), dtype=np.float32)
    rewards = np.zeros(len(actions), dtype=np.float32)
    lost = np.zeros(len(actions), dtype=bool)
    for k, action in enumerate(actions):
        dropped = game.drop(board, piece, action)
        if dropped is None:
            lost[k], rewards[k] = True, game.lose
        else:
            new, cleared = game.clear(dropped)
            X[k, :-1], X[k, -1], rewards[k] = new.ravel(), cleared, cleared
    return actions, X, rewards, lost


def q_values(net, X, rewards, lost):
    """Q-values of the placements: the network, or the reward for those that end the game."""
    with torch.no_grad():
        q = net(torch.from_numpy(X)).squeeze(1).numpy()
    return np.where(lost, rewards, q)

**Question 5.** Complete `dqn_loss(net, target_net, batch, gamma)`. A transition of the
buffer is `(x, r, next)`, where `x` is the input of the placement played, `r` its reward,
and `next` is `None` if the game ended, and otherwise the triple `(X, rewards, lost)` of the
placements of the next state. The loss is the mean of $(Q(x) - y)^2$ over the batch, where
$y$ is computed with the target network and is not differentiated.

In [ ]:
def dqn_loss(net, target_net, batch, gamma):
    x = torch.from_numpy(np.array([t[0] for t in batch]))
    ...  # TODO


def train_dqn(seed, n_games=300, gamma=0.95, epsilon=0.05, lr=1e-3, batch_size=32, sync=500):
    torch.manual_seed(seed)
    rng, game = np.random.default_rng(seed), Blocks(max_pieces=200, rng=seed)
    net = nn.Sequential(nn.Linear(61, 64), nn.ReLU(), nn.Linear(64, 1))
    target_net = copy.deepcopy(net)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    buffer, rows, step = [], [], 0
    for _ in range(n_games):
        (board, piece), done = game.reset(), False
        actions, X, rewards, lost = inputs(game, board, piece)
        while not done:
            q = q_values(net, X, rewards, lost)
            k = int(rng.integers(len(actions))) if rng.random() < epsilon else int(np.argmax(q))
            (board, piece), r, done = game.step(actions[k])
            nxt = None if done else inputs(game, board, piece)
            if not lost[k]:
                buffer.append((X[k], r, None if done else nxt[1:]))
                if len(buffer) > 20000:
                    buffer.pop(0)
            if not done:
                actions, X, rewards, lost = nxt
            step += 1
            if len(buffer) >= 500:
                batch = [buffer[i] for i in rng.integers(len(buffer), size=batch_size)]
                loss = dqn_loss(net, target_net, batch, gamma)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            if step % sync == 0:
                target_net.load_state_dict(net.state_dict())
        rows.append(game.rows)
    return np.array(rows)

**Question 6.** Train $2$ agents for $300$ games (about $2$ minutes each) and plot the mean
number of rows per game. Compare with linear Q-learning in Lab 8, which reached about $70$
rows after $200$ games.

In [ ]:
...  # TODO